# Step 1: Extract brandnames and codes from UKB lookups

In [ ]:
import sys
import json
from pprint import pprint

sys.path.append('../')
from prescriptions_processing import CodesExtractor

### Tutorial to CodesExtractor class

Start with using a constructor and `run_workflow()` function, which will give you dictionaries for all types of codes (bnf, dmd, read2).

1. Initialize an object using the `CodesExtractor` class with the specified output directory (where the dictionaries will be written), a specific `lkps_dir` (the directory where Lookups from UKBiobank were downloaded or where you want them to be downloaded; the default value for `lkps_dir` is a directory named `lkps` in `output_dir`) and a specific `file_prefix` for each output file (if given).

In [ ]:
processor = CodesExtractor(file_prefix=None, output_dir='../data/codes_lkps', lkps_dir='../data/ukb_lkps')

2. Load the substances from a file into a Python dictionary (`substances_dict`) for further processing. The dictionary uses unique substance names as keys, and the values are lists of alternative names for each substance, including the key name itself. The keys must only consist of numbers, letters, and the hyphen (-), with no other special characters.

In [ ]:
with open('../data/input/substances.json', 'r') as file:
    substances_dict = json.loads(file.read())

3. Load manual refinement rules (`brand_names_refinement_rules.json` file) for filtering drug brand name extracted from UKB dictionaries. Mostly ignoring some pharma companies names and adding a few omitted drugs brand names.

In [ ]:
with open('../data/input/brand_names_refinement_rules.json', 'r') as file:
    refinement_rules = json.loads(file.read())
pprint(refinement_rules)

4. Run workflow (extract all dictionaries) for given substances. The `run_workflow` function works in parallel for each type of code, extracting all dictionaries concurrently.

In [ ]:
%%time
processor.run_workflow(substances_dict, refinement_rules)

5. Check which substances were omitted (ignored) during substring filtering and refinement.

**Important notice**: Substring filtering – if applied – causes false negatives in prescription filtering. But without substring filtering, results include false positives.

In [ ]:
header = [(' #brand-name ', '#substance', '#where-matched', '#matching-method')]
print('\n'.join(['   '.join(record) for record in (header + processor.omitted_brand_names)]))

### Substances without codes

The variable `substances_without_codes` contains the list of substances that do not have codes in any of the dictionaries.  

There are 11 such substances:

- bempedoic
- bempedoic_acid
- cenobamate
- cytisinicline
- daridorexant
- finerenone
- fremanezumab
- icosapent
- inclisiran
- rimegepant
- vericiguat

These are substances that have been approved for sale in the UK since the last update of UK Biobank data (after 2017) or substances that are so rare that they have not been assigned to any individual in UK Biobank due to their lack of popularity in the UK.

In [ ]:
import json
with open('../data/codes_lkps/bnf_codes.json', 'r') as f:
    bnf_dictionary = json.load(f)
with open('../data/codes_lkps/read2_codes.json', 'r') as f:
    read_v2_dictionary = json.load(f)
with open('../data/codes_lkps/dmd_codes.json', 'r') as f:
    dmd_dictionary = json.load(f)

In [ ]:
substances_without_codes = []

for substance in substances_dict:
    if all(
        (substance in code_dict and isinstance(code_dict[substance], list) and len(code_dict[substance]) == 0)
        or (substance not in code_dict)
        for code_dict in [bnf_dictionary, read_v2_dictionary, dmd_dictionary]
    ):
        substances_without_codes.append(substance)

print("Substances without codes in all dictionaries:", substances_without_codes)